# Finite difference method to simulate Spinodal decomposition.

<!-- #### <p style="text-align: right;"> &#9989; **put your name here** </p> -->

<!-- **DUE on 11/3** -->

---

In this coding assignment, you will use finite difference method to simulate spinodal decomposition with Cahn-Hilliard equation.

Below is an image from the paper of '*Ultrahigh Kinetic Inductance Superconducting Materials from Spinodal Decomposition*', July 2022, *Advanced Materials* **34**(32), DOI: 10.1002/adma.202201268. It illustrates the spinodal decomposition of an Al-Ti alloy.

<div align="left">
<img src="https://advanced.onlinelibrary.wiley.com/cms/asset/40df45d3-198d-487d-b59c-4ca1506f6334/adma202201268-fig-0001-m.jpg" width="600">
</div>

## Part 1. Plot the free energy function of regular solution model.

The regular solution model leads to a free energy function as

$$f(X) = \frac{\alpha}{k_B T} X \big(1-X\big) + X \ln X + \big(1-X \big) \ln \big(1-X \big).$$

### Part 1.1. Plot the curve of energy function.

Plot the free energy function at various temperatures. Use `T = np.linspace(0.75,2,11)`. Here, we select $\alpha = 3$ and $k_B = 1$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


ak = 3

T = np.linspace(0.75,2,11)
X = np.linspace(1.0e-6, 1-1.0e-6,1001)

for Tk in T:
    f = (ak/Tk)*X*(1-X) + X*np.log(X) + (1-X)*np.log(1-X)
    plt.plot(X, f, label=fr"T = {Tk:.3g}")   # format how you like: .2f, .3g, etc.

plt.grid(True)
plt.legend(title="T", loc="best")  # or loc="upper right", ncol=2, etc.
plt.xlabel("X"); plt.ylabel("f")
plt.show()

### Part 1.2. Find the binodal points and spinodal points.

Here, let's use the curve of `T = 2`. 

* Plot the curve of $f$ at $T=2$.

* The chemical potential in the bulk phase is the derivative of free energy $f$ with respect to composition $X$. 

$$\frac{\partial f}{\partial X} = \frac{\alpha}{k_B T} \big(1-2X \big) + \ln X - \ln \big(1-X \big).$$

* Plot the curve of $\frac{\partial f}{\partial X}$. The minima of $f$ are at where $\frac{\partial f}{\partial X} = 0$. Estimate the values of $X$ for the binodal points (or phase boundaries, solubility limits).

In [ ]:
# Complete the code below

Tk = 1

f0 = (ak/Tk)*X*(1-X) + X*np.log(X) + (1-X)*np.log(1-X)
df = (ak/Tk)*(1-2*X) + np.log(X) - np.log(1-X)

# --- plot side-by-side ---
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True)
ax1, ax2 = axes

ax1.plot(X, f0, lw=2)
ax1.set_title(r"$f_0(X) = \frac{a_k}{T_k}X(1-X) + X\ln X + (1-X)\ln(1-X)$")
ax1.set_xlabel("X")
ax1.set_ylabel("f0")
ax1.grid(True, alpha=0.3)

ax2.plot(X, df, lw=2)
ax2.set_title(r"$\frac{df_0}{dX} = \frac{a_k}{T_k}(1-2X) + \ln X - \ln(1-X)$")
ax2.set_xlabel("X")
ax2.set_ylabel("df/dX")
ax2.set_ylim(-1, 1)
ax2.grid(True, alpha=0.3)


<!-- &#9989; Do This - Estimate the values of X at the binodal points.  -->

The spinodal points are the boundaries of metastable region, which are the inflection points on the free energy function. The inflection point is where the concavity changes, i.e., concave up changes to concave down. Or vice versa. Clearly, the inflection points are where the local maximum and local minimum are. They will be at where the derivative of $\frac{\partial f}{\partial X}$ is zero, i.e.,

$$\frac{\partial^2 f}{\partial X^2} = 0 $$

* Plot the curve of

$$\frac{\partial^2 f}{\partial X^2} = -2\frac{\alpha}{k_B T} + \frac{1}{X\big(1-X \big)}$$

* Since the value of $\frac{\partial^2 f}{\partial X^2}$ near $X = 0$ will be enormous, let's set a `plt.ylim(-2.5,2)`.

In [ ]:
# complete the code 

Tk = 1

ddf = -2*(ak/Tk) + 1/X + 1/(1-X)

plt.plot(X,ddf)
plt.xlim(0.01,0.99)
plt.ylim(-2.5,2)
plt.grid(True)

<!-- &#9989; Do This - Estimate the values of X at the spinodal points. -->

---
## Part 2. Spinodal decomposition simulations.

### Part 2.1. Spontaneous phase separation.

Based on the thermodynamic theory, spontaneous phase separation can occur only when the composition is in the spinodal regions (between the two sponidal points).

* Here, let's first set the initial composition to have an average around $\bar{X}$ = `0.4` and simulation a 2D phase separation using Cahn-Hilliard equation.

$$\frac{\partial X}{\partial t} = L \nabla^2 \mu ~\Longrightarrow~\frac{\partial X}{\partial t} = L \nabla^2 \bigg( \frac{\partial f}{\partial X} - K \nabla^2 X\bigg).$$

The stencil for the Laplacian is 

$$\nabla^2 X_{i,j} = \frac{X_{i,j-1} - 2X_{i,j} + X_{i,j+1}}{\Delta x^2} + \frac{X_{i-1,j} - 2X_{i,j} + X_{i+1,j}}{\Delta y^2} $$

$$\mu_{i,j} = \frac{\partial f}{\partial X}\bigg|_{i,j} -K\nabla^2 X_{i,j}$$

$$\nabla^2 \mu_{i,j} = \frac{\mu_{i,j-1} - 2\mu_{i,j} + \mu_{i,j+1}}{\Delta x^2} + \frac{\mu_{i-1,j} - 2\mu_{i,j} + \mu_{i+1,j}}{\Delta y^2} $$

$$X_{i,j}^{(n+1)} = X_{i,j}^{(n)} + \Delta t L \nabla^2 \mu_{i,j}^{n}.$$ 

* Here, we will use periodic boundary conditions.
* We will use an initial condition that $\phi$ is randomly distributed between `0.3` and `0.5`.
* Note that since we will take Laplacian of $\mu$, we need to include boundary condtions for $\mu$.
* Follow the Allen-Cahn equation example code to complete the code for spinodal decomposition.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import time

# --- params ---
Nx = 128*2
Ny = Nx

dx = 1.0
dt = 0.01

W = 1;
K = 1.44;
L = 1.0

ak = 3
Tk = 1

# --- arrays / ICs ---
phi = np.random.uniform(0.3, 0.5, size=(Nx, Ny))   # in (0.3, 0.5)
mu  = np.zeros((Nx,Ny))
Lap = np.zeros((Nx,Ny))
tm  = 0.0


from IPython.display import display, clear_output
import time

vmin, vmax = -0., 1.   # set to what you want

fig, ax = plt.subplots(figsize=(5,5))
im = ax.imshow(phi, origin='upper', interpolation='nearest',
               vmin=vmin, vmax=vmax, cmap='viridis')   # <- caxis via vmin/vmax
cbar = fig.colorbar(im, ax=ax, label='phi')            # <- colorbar once
ax.set_aspect('equal', adjustable='box')
ax.set_xlim(0, Nx)
ax.set_ylim(0, Ny)


# --- time evolution ---
for iter in range(1, 10002):

    # vectorized computing
    # Laplace
    Lap[1:-1,1:-1] = (phi[:-2,1:-1] - 2.0*phi[1:-1,1:-1] + phi[2:,1:-1]) / (dx*dx) + \
                     (phi[1:-1,:-2] - 2.0*phi[1:-1,1:-1] + phi[1:-1,2:]) / (dx*dx) 

    # chemical potential (interior)
    fprime = (ak/Tk) * (1.0-2*phi[1:-1,1:-1])+ np.log(phi[1:-1,1:-1]) - np.log(1.0-phi[1:-1,1:-1]) 
    mu[1:-1,1:-1] = W * fprime - K * Lap[1:-1,1:-1]
    
    # periodic ghost-cell style BCs
    mu[0,1:-1]  = mu[-2,1:-1]
    mu[-1,1:-1] = mu[1,1:-1]
    mu[1:-1,0]  = mu[1:-1,-2]
    mu[1:-1,-1] = mu[1:-1,1]

    # Laplace
    Lap[1:-1,1:-1] = (mu[:-2,1:-1] - 2.0*mu[1:-1,1:-1] + mu[2:,1:-1]) / (dx*dx) + \
                     (mu[1:-1,:-2] - 2.0*mu[1:-1,1:-1] + mu[1:-1,2:]) / (dx*dx)    
    
    # explicit update
    phi[1:-1,1:-1] += dt * L * Lap[1:-1,1:-1]
    

    # periodic ghost-cell style BCs
    phi[0,1:-1]  = phi[-2,1:-1]
    phi[-1,1:-1] = phi[1,1:-1]
    phi[1:-1,0]  = phi[1:-1,-2]
    phi[1:-1,-1] = phi[1:-1,1]
    

    tm += dt

    # visualization
    if iter % 100 == 1:
        im.set_data(phi)                 # update image only
        ax.set_title(f"{iter}")
        
        # Animaiton part (dosn't change)
        clear_output(wait=True) # Clear output for dynamic display
        display(fig)            # Reset display
        # fig.clear()             # Prevent overlapping and layered plots
        time.sleep(0.0002)         # Sleep for half a second to slow down the animation  

### Part 2.2. Metastable region

According to the thermodynamic theory, spontaneous will not occur when the composition in the metastable region, i.e., $X$ between binodal and spinodal points. 

* Copy your functioning code to the cell below.
* Set an initial condition that the average of $\bar{X}$ is 1.7. Set $X$ to be randomly distributed between `0.07` and `0.27`.
* Rerun the simulation to check whether spinodal decomposition occurs.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import time

# --- params ---
Nx = 128*2
Ny = Nx

dx = 1.0
dt = 0.01

W = 1;
K = 1.44;
L = 1.0

ak = 3
Tk = 1

# --- arrays / ICs ---
phi = np.random.uniform(0.07, 0.27, size=(Nx, Ny))   # in (0.3, 0.5)
mu  = np.zeros((Nx,Ny))
Lap = np.zeros((Nx,Ny))
tm  = 0.0


from IPython.display import display, clear_output
import time

vmin, vmax = -0., 1.   # set to what you want

fig, ax = plt.subplots(figsize=(5,5))
im = ax.imshow(phi, origin='upper', interpolation='nearest',
               vmin=vmin, vmax=vmax, cmap='viridis')   # <- caxis via vmin/vmax
cbar = fig.colorbar(im, ax=ax, label='phi')            # <- colorbar once
ax.set_aspect('equal', adjustable='box')
ax.set_xlim(0, Nx)
ax.set_ylim(0, Ny)


# --- time evolution ---
for iter in range(1, 10002):

            
    # vectorized computing
    # Laplace
    Lap[1:-1,1:-1] = (phi[:-2,1:-1] - 2.0*phi[1:-1,1:-1] + phi[2:,1:-1]) / (dx*dx) + \
                     (phi[1:-1,:-2] - 2.0*phi[1:-1,1:-1] + phi[1:-1,2:]) / (dx*dx) 

    # chemical potential (interior)
    fprime = (ak/Tk) * (1.0-2*phi[1:-1,1:-1])+ np.log(phi[1:-1,1:-1]) - np.log(1.0-phi[1:-1,1:-1]) 
    mu[1:-1,1:-1] = W * fprime - K * Lap[1:-1,1:-1]
    
    # periodic ghost-cell style BCs
    mu[0,1:-1]  = mu[-2,1:-1]
    mu[-1,1:-1] = mu[1,1:-1]
    mu[1:-1,0]  = mu[1:-1,-2]
    mu[1:-1,-1] = mu[1:-1,1]

    # Laplace
    Lap[1:-1,1:-1] = (mu[:-2,1:-1] - 2.0*mu[1:-1,1:-1] + mu[2:,1:-1]) / (dx*dx) + \
                     (mu[1:-1,:-2] - 2.0*mu[1:-1,1:-1] + mu[1:-1,2:]) / (dx*dx)    
    
    # explicit update
    phi[1:-1,1:-1] += dt * L * Lap[1:-1,1:-1]
    

    # periodic ghost-cell style BCs
    phi[0,1:-1]  = phi[-2,1:-1]
    phi[-1,1:-1] = phi[1,1:-1]
    phi[1:-1,0]  = phi[1:-1,-2]
    phi[1:-1,-1] = phi[1:-1,1]
    

    tm += dt

    # visualization
    if iter % 100 == 1:
        im.set_data(phi)                 # update image only
        ax.set_title(f"{iter}")
        
        # Animaiton part (dosn't change)
        clear_output(wait=True) # Clear output for dynamic display
        display(fig)            # Reset display
        # fig.clear()             # Prevent overlapping and layered plots
        time.sleep(0.0002)         # Sleep for half a second to slow down the animation  

<!-- &#9989; Do This - Compare and discuss the results of $\bar{X}=0.4$ and $\bar{X} = 0.17$. -->

---
### Part 2.3. Growth from a nucleus.

In this test, let's start with an initial average composition to be `0.12` throughout the computational box. Without a nucleus, two phase coexistence will not occur.

Below is an figure illustrating spinodal decomposition and nucleation growth. In the left panel of figure, it shows a a nucleation growth mechanism. In nucleation growth, nuclei grow by absorbing concentration from the surrounding environment. In conntrast, the right panel shows a spontaneous spinodal composition (phase separation) process, during which **uphill diffusion** can occur during spinodal decomposition. 

<!-- <div align="left">
<img src="https://ibb.co/CKtL97mk" width="800">
</div> -->

<img src="https://i.ibb.co/jk5twMbc/Spinodal-decomposition.png" width="800" alt="Spinodal-decomposition" border="0" />

image from D.A. Porter, K.E. Easterling, M. Sherif, '*Phase transformation in metals and alloys*', CRC press, 2009.

* Run a simulation with initial $\bar{X} = 0.12$ first. Observe whether spinodal decomposition will occur.

* Add a seed (nucleus) that has a radius of $r=5$ at `(100,100)`. In this case, the average composition will be $\bar{X} = \pi r^2 /256^2 = 0.0012 +0.12$, which is still well below the spinodal point.

* Run the simulation to examine whether the seed will growth.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import time

# --- params ---
Nx = 128*2
Ny = Nx

dx = 1.0
dt = 0.01

W = 1;
K = 1.44;
L = 1.0

ak = 3
Tk = 1

# --- arrays / ICs ---
mu  = np.zeros((Nx,Ny))
phi = np.random.uniform(0.02, 0.22, size=(Nx, Ny))   # in (0.02, 0.22)
Lap = np.zeros((Nx,Ny))
tm  = 0.0

for p in range(Nx):
    for q in range(Ny):
        rad = np.sqrt((p*dx-100)**2+(q*dx-100)**2)
        if rad - 5 < 0:
            phi[p,q] = 0.91



from IPython.display import display, clear_output
import time

vmin, vmax = 0.08, 0.15   # set to what you want

fig, ax = plt.subplots(figsize=(5,5))
im = ax.imshow(phi, origin='upper', interpolation='nearest',
               vmin=vmin, vmax=vmax, cmap='viridis')   # <- caxis via vmin/vmax
cbar = fig.colorbar(im, ax=ax, label='phi')            # <- colorbar once
ax.set_aspect('equal', adjustable='box')
ax.set_xlim(0, Nx)
ax.set_ylim(0, Ny)


# --- time evolution ---
for iter in range(1, 50002):

            
    # vectorized computing
    # Laplace
    Lap[1:-1,1:-1] = (phi[:-2,1:-1] - 2.0*phi[1:-1,1:-1] + phi[2:,1:-1]) / (dx*dx) + \
                     (phi[1:-1,:-2] - 2.0*phi[1:-1,1:-1] + phi[1:-1,2:]) / (dx*dx) 

    # chemical potential (interior)
    fprime = (ak/Tk) * (1.0-2*phi[1:-1,1:-1])+ np.log(phi[1:-1,1:-1]) - np.log(1.0-phi[1:-1,1:-1]) 
    mu[1:-1,1:-1] = W * fprime - K * Lap[1:-1,1:-1]
    
    # periodic ghost-cell style BCs
    mu[0,1:-1]  = mu[-2,1:-1]
    mu[-1,1:-1] = mu[1,1:-1]
    mu[1:-1,0]  = mu[1:-1,-2]
    mu[1:-1,-1] = mu[1:-1,1]

    # Laplace
    Lap[1:-1,1:-1] = (mu[:-2,1:-1] - 2.0*mu[1:-1,1:-1] + mu[2:,1:-1]) / (dx*dx) + \
                     (mu[1:-1,:-2] - 2.0*mu[1:-1,1:-1] + mu[1:-1,2:]) / (dx*dx)    
    
    # explicit update
    phi[1:-1,1:-1] += dt * L * Lap[1:-1,1:-1]
    

    # periodic ghost-cell style BCs
    phi[0,1:-1]  = phi[-2,1:-1]
    phi[-1,1:-1] = phi[1,1:-1]
    phi[1:-1,0]  = phi[1:-1,-2]
    phi[1:-1,-1] = phi[1:-1,1]
    

    tm += dt

    # visualization
    if iter % 100 == 1:
        im.set_data(phi)                 # update image only
        ax.set_title(f"{iter}")
        
        
        # Animaiton part (dosn't change)
        clear_output(wait=True) # Clear output for dynamic display
        display(fig)            # Reset display
        # fig.clear()             # Prevent overlapping and layered plots
        time.sleep(0.0002)         # Sleep for half a second to slow down the animation  

<!-- &#9989; Do This - Descibe your observations. -->

---
## Part 3. Temperature effect

Now, let's examine the effect of temperature. According the results in Part 1.1, if $T=1.75$, the energy function is a single-well function. This suggests that uniform mixture is thermodynamically favored.

* Copy your function code of Part 2.1 to the cell below.
* Change temperature to $T=1.75$.
* Increase the amplitude of the initial noise to be larger: e.g., random distribution between 0.05 and 0.75 such that the average is stll $\bar{X} = 0.4$.
* Run the simulation.

In [ ]:
# put your code here.

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import time

# --- params ---
Nx = 128*2
Ny = Nx

dx = 1.0
dt = 0.01

W = 1;
K = 1.44;
L = 1.0

ak = 3
Tk = 1.75

# --- arrays / ICs ---
mu  = np.zeros((Nx,Ny))
phi = np.random.uniform(0.05, 0.85, size=(Nx, Ny))   # in (0.05, 0.85)
Lap = np.zeros((Nx,Ny))
tm  = 0.0


from IPython.display import display, clear_output
import time

vmin, vmax = -0., 1.   # set to what you want

fig, ax = plt.subplots(figsize=(5,5))
im = ax.imshow(phi, origin='upper', interpolation='nearest',
               vmin=vmin, vmax=vmax, cmap='viridis')   # <- caxis via vmin/vmax
cbar = fig.colorbar(im, ax=ax, label='phi')            # <- colorbar once
ax.set_aspect('equal', adjustable='box')
ax.set_xlim(0, Nx)
ax.set_ylim(0, Ny)


# --- time evolution ---
for iter in range(1, 10002):

            
    # vectorized computing
    # Laplace
    Lap[1:-1,1:-1] = (phi[:-2,1:-1] - 2.0*phi[1:-1,1:-1] + phi[2:,1:-1]) / (dx*dx) + \
                     (phi[1:-1,:-2] - 2.0*phi[1:-1,1:-1] + phi[1:-1,2:]) / (dx*dx) 

    # chemical potential (interior)
    fprime = (ak/Tk) * (1.0-2*phi[1:-1,1:-1])+ np.log(phi[1:-1,1:-1]) - np.log(1.0-phi[1:-1,1:-1]) 
    mu[1:-1,1:-1] = W * fprime - K * Lap[1:-1,1:-1]
    
    # periodic ghost-cell style BCs
    mu[0,1:-1]  = mu[-2,1:-1]
    mu[-1,1:-1] = mu[1,1:-1]
    mu[1:-1,0]  = mu[1:-1,-2]
    mu[1:-1,-1] = mu[1:-1,1]

    # Laplace
    Lap[1:-1,1:-1] = (mu[:-2,1:-1] - 2.0*mu[1:-1,1:-1] + mu[2:,1:-1]) / (dx*dx) + \
                     (mu[1:-1,:-2] - 2.0*mu[1:-1,1:-1] + mu[1:-1,2:]) / (dx*dx)    
    
    # explicit update
    phi[1:-1,1:-1] += dt * L * Lap[1:-1,1:-1]
    

    # periodic ghost-cell style BCs
    phi[0,1:-1]  = phi[-2,1:-1]
    phi[-1,1:-1] = phi[1,1:-1]
    phi[1:-1,0]  = phi[1:-1,-2]
    phi[1:-1,-1] = phi[1:-1,1]
    

    tm += dt

    # visualization
    if iter % 100 == 1:
        im.set_data(phi)                 # update image only
        ax.set_title(f"{iter}")
        
        # Animaiton part (dosn't change)
        clear_output(wait=True) # Clear output for dynamic display
        display(fig)            # Reset display
        # fig.clear()             # Prevent overlapping and layered plots
        time.sleep(0.0002)         # Sleep for half a second to slow down the animation  
        

<!-- &#9989; Do This - Descibe your observations. -->

<!-- ---
### Great! You're done! Please add your name to the file name and upload your file to the respective drop box. This assignment is due on 11/3.  -->